# ComfyUI + Cloudflare Tunnel — Minimal Test

This notebook is **only for testing the tunnel**.

It does **not** download H3, LTX, Director models, or workflows. It starts a bare ComfyUI instance on `127.0.0.1:8188`, verifies the local origin, starts your named Cloudflare Tunnel, and shows the diagnostic logs.

Before running:

- Cloudflare published application / tunnel route must point `comfy.zetbros.com` to **`http://127.0.0.1:8188`**.
- Colab Secret **`CF_TUNNEL_TOKEN`** must contain the tunnel token. The cell below also accepts the full `cloudflared ... --token eyJ...` or `cloudflared service install eyJ...` command and extracts the token safely.


## 1. Read the Cloudflare tunnel token from Colab Secrets

In [ ]:
from google.colab import userdata
import re, hashlib

raw = userdata.get('CF_TUNNEL_TOKEN')
if not raw:
    raise RuntimeError('Missing CF_TUNNEL_TOKEN in Colab Secrets.')

raw = raw.strip()
match = re.search(r'(eyJ[A-Za-z0-9._-]+)', raw)
if not match:
    raise RuntimeError('CF_TUNNEL_TOKEN does not contain a Cloudflare tunnel token beginning with eyJ.')

CF_TOKEN = match.group(1)
print('✅ Tunnel token found')
print('Length:', len(CF_TOKEN))
print('Fingerprint:', hashlib.sha256(CF_TOKEN.encode()).hexdigest()[:12])
print('Token itself is intentionally not printed.')


## 2. Install a bare ComfyUI

This is intentionally model-free. We only need the web server to prove Cloudflare can reach it.

In [ ]:
import os, subprocess, sys
from pathlib import Path

COMFY = Path('/content/ComfyUI-Tunnel-Test')

if not (COMFY / '.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',str(COMFY)], check=True)
else:
    subprocess.run(['git','-C',str(COMFY),'pull','--ff-only'], check=False)

subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(COMFY/'requirements.txt')], check=True)
print('✅ Bare ComfyUI installed at', COMFY)


## 3. Start ComfyUI on IPv4 localhost only

In [ ]:
import os, subprocess, time, signal
from pathlib import Path

LOG = Path('/content/comfy-cloudflare-test')
LOG.mkdir(exist_ok=True)
COMFY_LOG = LOG / 'comfyui.log'

# Stop only an earlier copy of this dedicated test server.
subprocess.run(
    "pkill -f 'ComfyUI-Tunnel-Test/main.py.*--port 8188' || true",
    shell=True, check=False
)

with open(COMFY_LOG, 'w') as f:
    proc = subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--disable-auto-launch', '--cpu'],
        cwd=str(COMFY),
        stdout=f, stderr=subprocess.STDOUT,
        start_new_session=True
    )

for i in range(90):
    r = subprocess.run(['curl','-fsS','--max-time','3','http://127.0.0.1:8188/system_stats'], capture_output=True, text=True)
    if r.returncode == 0:
        print('✅ ComfyUI local origin is healthy: http://127.0.0.1:8188')
        break
    if proc.poll() is not None:
        print(COMFY_LOG.read_text()[-6000:])
        raise RuntimeError(f'ComfyUI exited early with code {proc.returncode}')
    time.sleep(2)
else:
    print(COMFY_LOG.read_text()[-6000:])
    raise RuntimeError('ComfyUI did not become healthy within 180 seconds.')

print('PID:', proc.pid)


## 4. Verify the exact local origin Cloudflare should use

In [ ]:
import subprocess

print('--- /system_stats through 127.0.0.1 ---')
r = subprocess.run(['curl','-sS','-o','/tmp/comfy_stats.json','-w','HTTP %{http_code}\n','http://127.0.0.1:8188/system_stats'], capture_output=True, text=True)
print(r.stdout.strip())
print(open('/tmp/comfy_stats.json').read()[:500])

print('\n--- listening socket ---')
print(subprocess.run("ss -ltnp | grep ':8188' || true", shell=True, capture_output=True, text=True).stdout)

print('\nCloudflare route MUST be: http://127.0.0.1:8188')


## 5. Install cloudflared

In [ ]:
import platform, subprocess, os

arch = platform.machine().lower()
cf_arch = 'amd64' if arch in ('x86_64','amd64') else 'arm64' if arch in ('aarch64','arm64') else None
if not cf_arch:
    raise RuntimeError(f'Unsupported architecture: {arch}')

if subprocess.run(['bash','-lc','command -v cloudflared >/dev/null 2>&1']).returncode != 0:
    url = f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}'
    subprocess.run(['curl','-fL','--retry','3','--retry-delay','2',url,'-o','/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod','0755','/usr/local/bin/cloudflared'], check=True)

print(subprocess.check_output(['cloudflared','--version'], text=True).strip())


## 6. Start the named Cloudflare Tunnel connector

In [ ]:
import subprocess, time, os
from pathlib import Path

CF_LOG = Path('/content/comfy-cloudflare-test/cloudflared.log')
subprocess.run("pkill -f 'cloudflared.*tunnel.*run' || true", shell=True, check=False)

with open(CF_LOG, 'w') as f:
    cf = subprocess.Popen(
        ['cloudflared','tunnel','--no-autoupdate','run','--token',CF_TOKEN],
        stdout=f, stderr=subprocess.STDOUT,
        start_new_session=True
    )

registered = False
for i in range(60):
    if cf.poll() is not None:
        print(CF_LOG.read_text()[-6000:])
        raise RuntimeError(f'cloudflared exited with code {cf.returncode}')
    text = CF_LOG.read_text(errors='ignore') if CF_LOG.exists() else ''
    if 'Registered tunnel connection' in text or 'Connection' in text and 'registered' in text.lower():
        registered = True
        break
    time.sleep(1)

print('cloudflared PID:', cf.pid)
print('Connector registration detected:', registered)
print('\n--- last cloudflared log lines ---')
print('\n'.join(CF_LOG.read_text(errors='ignore').splitlines()[-30:]))

if not registered:
    print('\n⚠️ cloudflared is still running, but registration was not positively detected yet.')
else:
    print('\n✅ Cloudflare connector is registered.')


## 7. Final checklist

If Sections 3, 4, and 6 are green:

1. In Cloudflare, confirm the published application for `comfy.zetbros.com` uses **HTTP** and origin **`127.0.0.1:8188`**.
2. Open `https://comfy.zetbros.com` in your browser and complete Cloudflare Access login.
3. You should see the bare ComfyUI interface. Missing models are expected in this test notebook.

If the browser still shows **502**, run Section 8 below and send the output.


## 8. Diagnostics if the browser still shows 502

In [ ]:
from pathlib import Path
import subprocess

print('=== LOCAL COMFY ===')
print(subprocess.run(['curl','-sS','-o','/dev/null','-w','HTTP %{http_code}\n','http://127.0.0.1:8188/system_stats'], capture_output=True, text=True).stdout.strip())
print(subprocess.run("ss -ltnp | grep ':8188' || true", shell=True, capture_output=True, text=True).stdout)

print('=== CLOUDFLARED PROCESS ===')
print(subprocess.run("pgrep -af 'cloudflared.*tunnel.*run' || true", shell=True, capture_output=True, text=True).stdout)

print('=== CLOUDFLARED LOG ===')
cf_log = Path('/content/comfy-cloudflare-test/cloudflared.log')
print('\n'.join(cf_log.read_text(errors='ignore').splitlines()[-80:]) if cf_log.exists() else 'No cloudflared log found')

print('=== COMFY LOG ===')
comfy_log = Path('/content/comfy-cloudflare-test/comfyui.log')
print('\n'.join(comfy_log.read_text(errors='ignore').splitlines()[-50:]) if comfy_log.exists() else 'No ComfyUI log found')
